# RATA-MAR — Revision 1 (IASC-94359): GPU runs on A100

**Runtime → Change runtime type → A100 GPU**, then **Runtime → Run all**. Everything is written to
`REVISION_R1/experiments/outputs/` on Google Drive, and every step resumes if the runtime disconnects (run all again).

| Step | What | Approx. time (A100) |
|---|---|---|
| 1 | Inference-only analyses on the **published** checkpoints (reproduction check, statistics, extended stress tests, delay sweep, detection) → `outputs/cuda/` | 10–20 min |
| 2 | Two inference diagnostics (oracle gate, gap filling) → `outputs/cuda/` | 2–5 min |
| 3 | Additional training (25 runs × 100 epochs): R0 reproduction, token count K, reliability targets, TMC-/QMF-style baselines → `outputs/gpu_train/` | 1–3 h |
| 4 | Regenerate figures and LaTeX tables from the GPU results | < 1 min |

The published checkpoints in `runs/RATA_MAR_SI_v21/` are only **read**, never modified.
When finished, all results are in `REVISION_R1/experiments/outputs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/RATA_IASC/REVISION_R1/experiments')
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select an A100 GPU runtime first.'
!pip -q install scikit-learn pandas matplotlib

In [ ]:
# Step 1 — inference-only analyses on GPU (identical procedure to the paper's evaluation)
!python -u r1_inference_analyses.py --stages ABCDE 2>&1 | grep -v -i warning | tee outputs/log_inference_cuda.txt | tail -25

In [ ]:
# Reproduction check against the submitted paper: deterministic AND stochastic scenarios should now match exactly
import pandas as pd
chk = pd.read_csv('outputs/cuda/A_reproduction_check.csv')
print('max |diff| (percentage points) by metric:'); print(chk.groupby('metric').abs_diff_pp.max().round(3))

In [ ]:
# Step 2 — inference diagnostics (Comment 12: oracle gate; Comments 3/8: gap filling)
!python -u r1_oracle_gate.py 2>&1 | grep -v -i warning | tail -8
!python -u r1_gap_diagnostic.py 2>&1 | grep -v -i warning | tail -12

In [ ]:
# Step 3 — additional training (resumable; just re-run this cell after a disconnect)
!python -u r1_train_variants_gpu.py --groups R0,K,T,B 2>&1 | grep -v -i warning | tee -a outputs/log_train_gpu.txt | tail -60

In [ ]:
# Step 4 — figures and LaTeX tables from the GPU results + summary
!python -u r1_figures.py --device cuda
!python -u r1_tables.py
import pandas as pd
d = pd.read_csv('outputs/gpu_train/variants_per_seed.csv')
cols = ['clean_top1','missing_skeleton','missing_imu','skeleton_noise','imu_noise','joint_drop_30','delay_pm16','extended_mean_top1']
pd.set_option('display.width', 250)
print((d.groupby('variant')[cols].agg(['mean','std'])*100).round(2).to_string())
print(open('outputs/gpu_train/R0_reproduction.txt').read() if os.path.exists('outputs/gpu_train/R0_reproduction.txt') else '')
print('Done.')